# 2D FDTD — Shared Memory Experiment (1000x1000)

**Authors:** Vraj Patel (241110080), Vardhman Dwivedi (241060033)  
**Course:** IDC 606 — Fast Computational Hydrodynamics, IIT Kanpur

### Purpose
At 200x200 (469 KB), the entire working set fits in L2 cache, so shared memory gives no benefit.  
At **1000x1000 (11.7 MB)**, the working set exceeds L2 (4 MB), so shared memory should help by caching tiles.

### Versions compared
1. **CuPy Global Memory** — same RawKernel as the main notebook, no shared memory
2. **CuPy Shared Memory** — tiles loaded into shared memory with halo, `__syncthreads()` barrier
3. **CUDA C Global Memory** — nvcc compiled, no shared memory
4. **CUDA C Shared Memory** — nvcc compiled, shared memory tiling

All versions use block `(8, 32)`, 100 time steps, float32, no snapshots (pure compute benchmark).

In [ ]:
# Detect CUDA version and install matching CuPy
import subprocess
try:
    out = subprocess.check_output(['nvidia-smi'], text=True)
    print(out)
    # Extract CUDA version from nvidia-smi
    import re
    m = re.search(r'CUDA Version:\s+(\d+)\.(\d+)', out)
    if m:
        cuda_major = int(m.group(1))
        print(f'Detected CUDA {cuda_major}.{m.group(2)}')
        if cuda_major >= 12:
            !pip install cupy-cuda12x -q
        elif cuda_major == 11:
            !pip install cupy-cuda11x -q
        else:
            print(f'Unsupported CUDA {cuda_major}, trying generic cupy')
            !pip install cupy -q
except Exception as e:
    print(f'nvidia-smi failed: {e}')
    print('No GPU detected — CUDA C benchmarks will still work via nvcc')

In [ ]:
import numpy as np
import time

try:
    import cupy as cp
    HAS_CUPY = True
    print(f'CuPy {cp.__version__}, CUDA {cp.cuda.runtime.runtimeGetVersion()}')
except Exception as e:
    HAS_CUPY = False
    print(f'CuPy not available: {e}')
    print('Will skip CuPy benchmarks, CUDA C benchmarks still run.')

# ═══════════════════════════════════════════════════════════════
#  Grid & Physics Parameters (1000x1000)
# ═══════════════════════════════════════════════════════════════
c0   = 3.0e8
mu0  = 4.0e-7 * np.pi
eps0 = 1.0 / (mu0 * c0**2)

Nx = 1000
Ny = 1000
dx = 1e-3
dy = 1e-3

courant = 0.99
dt = courant / (c0 * np.sqrt(1.0/dx**2 + 1.0/dy**2))
n_steps = 100  # enough to benchmark, not too long

src_x, src_y = Nx // 2, Ny // 2
spread = 12.0 * dt
t0 = 3.0 * spread

coeff_hx = np.float32(dt / (mu0 * dy))
coeff_hy = np.float32(dt / (mu0 * dx))
coeff_ez = np.float32(dt / eps0)

working_set = 3 * Nx * Ny * 4
print(f'Grid         : {Nx} x {Ny} ({Nx*Ny:,} cells)')
print(f'Time steps   : {n_steps}')
print(f'Working set  : {working_set/1024:.0f} KB = {working_set/1024/1024:.1f} MB')
print(f'L2 cache     : ~4 MB')
print(f'Exceeds L2   : {"YES" if working_set > 4*1024*1024 else "NO"} — shared memory matters here')

# FLOP counting
flops_per_step = Nx*(Ny-1)*3 + (Nx-1)*Ny*3 + (Nx-1)*(Ny-1)*7
flops_total = flops_per_step * n_steps
print(f'FLOPs/step   : {flops_per_step:,}')
print(f'Total FLOPs  : {flops_total:,} ({flops_total/1e9:.2f} GFLOP)')

# Pre-compute pulse values
pulse_np = np.array([np.exp(-((n*dt - t0)/spread)**2)
                     for n in range(n_steps)], dtype=np.float32)
pulse_gpu = cp.array(pulse_np, dtype=cp.float32) if HAS_CUPY else None

# Block/Grid config
BLOCK = (8, 32) if HAS_CUPY else None
GRID  = ((Nx + 8 - 1) // 8, (Ny + 32 - 1) // 32) if HAS_CUPY else None
print(f'Block        : {BLOCK} = {BLOCK[0]*BLOCK[1]} threads')
print(f'Grid         : {GRID} = {GRID[0]*GRID[1]} blocks')

## Version 1: CuPy Global Memory (no shared memory)

Same kernels as the main notebook — each thread reads directly from global memory.

In [ ]:
if not HAS_CUPY:
    print('Skipping CuPy global kernels (no CuPy)')
else:
    # ═══════════════════════════════════════════════════════════════
    #  Global Memory Kernels (baseline)
    # ═══════════════════════════════════════════════════════════════

    cupy_H_global = cp.RawKernel(r'''
    extern "C" __global__
    void update_H_global(float* Ez, float* Hx, float* Hy,
                         float coeff_hx, float coeff_hy,
                         int Nx, int Ny)
    {
        int i = blockIdx.x * blockDim.x + threadIdx.x;
        int j = blockIdx.y * blockDim.y + threadIdx.y;

        if (i < Nx && j < Ny - 1) {
            int idx = i * Ny + j;
            Hx[idx] -= coeff_hx * (Ez[idx + 1] - Ez[idx]);
        }
        if (i < Nx - 1 && j < Ny) {
            int idx = i * Ny + j;
            Hy[idx] += coeff_hy * (Ez[idx + Ny] - Ez[idx]);
        }
    }
    ''', 'update_H_global')


    cupy_Ez_global = cp.RawKernel(r'''
    extern "C" __global__
    void update_Ez_global(float* Ez, const float* Hx, const float* Hy,
                          float coeff_ez, float dx, float dy,
                          int Nx, int Ny,
                          int src_x, int src_y,
                          const float* pulse_arr, int step)
    {
        int i = blockIdx.x * blockDim.x + threadIdx.x;
        int j = blockIdx.y * blockDim.y + threadIdx.y;
        if (i >= Nx || j >= Ny) return;
        int idx = i * Ny + j;

        if (i >= 1 && j >= 1) {
            float dHy = (Hy[idx] - Hy[idx - Ny]) / dx;
            float dHx = (Hx[idx] - Hx[idx - 1])  / dy;
            Ez[idx] += coeff_ez * (dHy - dHx);
        }
        if (i == src_x && j == src_y)
            Ez[idx] = pulse_arr[step];
        if (i == 0 || i == Nx-1 || j == 0 || j == Ny-1)
            Ez[idx] = 0.0f;
    }
    ''', 'update_Ez_global')

    print('Global memory kernels compiled.')

## Version 2: CuPy Shared Memory

### How it works
1. Each block loads a **(BLOCK_X+2) x (BLOCK_Y+2)** tile into shared memory — the +2 is the 1-cell halo on each side
2. `__syncthreads()` ensures all threads finish loading before any thread reads
3. Interior threads read from shared memory (fast, ~100x vs DRAM)
4. Halo cells come from neighboring blocks' global memory

### Why this helps at 1000x1000
- Working set = 11.7 MB >> L2 cache (4 MB)
- Without shared memory: redundant DRAM reads for overlapping neighbor accesses
- With shared memory: each value loaded from DRAM once, read from SMEM multiple times

In [ ]:
if not HAS_CUPY:
    print('Skipping CuPy shared kernels (no CuPy)')
else:
    # ═══════════════════════════════════════════════════════════════
    #  Shared Memory Kernels
    # ═══════════════════════════════════════════════════════════════

    # H kernel with shared memory for Ez tile
    cupy_H_shared = cp.RawKernel(r'''
    #define BX 8
    #define BY 32

    extern "C" __global__
    void update_H_shared(float* Ez, float* Hx, float* Hy,
                         float coeff_hx, float coeff_hy,
                         int Nx, int Ny)
    {
        // Shared tile: (BX+1) x (BY+1) to hold the +1 neighbor in each direction
        // Hx needs Ez[i,j] and Ez[i,j+1] -> need +1 in j
        // Hy needs Ez[i,j] and Ez[i+1,j] -> need +1 in i
        __shared__ float s_Ez[(BX+1) * (BY+1)];

        int tx = threadIdx.x;  // 0..BX-1
        int ty = threadIdx.y;  // 0..BY-1
        int i = blockIdx.x * BX + tx;
        int j = blockIdx.y * BY + ty;

        // Shared memory pitch
        int sp = BY + 1;

        // Load main tile
        if (i < Nx && j < Ny)
            s_Ez[tx * sp + ty] = Ez[i * Ny + j];
        else
            s_Ez[tx * sp + ty] = 0.0f;

        // Load +1 halo in j (rightmost column threads load j+1)
        if (ty == BY - 1 || j == Ny - 1) {
            int jj = j + 1;
            if (i < Nx && jj < Ny)
                s_Ez[tx * sp + ty + 1] = Ez[i * Ny + jj];
            else
                s_Ez[tx * sp + ty + 1] = 0.0f;
        }

        // Load +1 halo in i (bottom row threads load i+1)
        if (tx == BX - 1 || i == Nx - 1) {
            int ii = i + 1;
            if (ii < Nx && j < Ny)
                s_Ez[(tx + 1) * sp + ty] = Ez[ii * Ny + j];
            else
                s_Ez[(tx + 1) * sp + ty] = 0.0f;
        }

        __syncthreads();

        // Hx update: needs Ez[i,j] and Ez[i,j+1]
        if (i < Nx && j < Ny - 1) {
            int idx = i * Ny + j;
            Hx[idx] -= coeff_hx * (s_Ez[tx * sp + ty + 1] - s_Ez[tx * sp + ty]);
        }

        // Hy update: needs Ez[i,j] and Ez[i+1,j]
        if (i < Nx - 1 && j < Ny) {
            int idx = i * Ny + j;
            Hy[idx] += coeff_hy * (s_Ez[(tx + 1) * sp + ty] - s_Ez[tx * sp + ty]);
        }
    }
    ''', 'update_H_shared')


    # Ez kernel with shared memory for Hx and Hy tiles
    cupy_Ez_shared = cp.RawKernel(r'''
    #define BX 8
    #define BY 32

    extern "C" __global__
    void update_Ez_shared(float* Ez, const float* Hx, const float* Hy,
                          float coeff_ez, float dx, float dy,
                          int Nx, int Ny,
                          int src_x, int src_y,
                          const float* pulse_arr, int step)
    {
        // Ez[i,j] needs: Hy[i,j], Hy[i-1,j], Hx[i,j], Hx[i,j-1]
        // So we need a -1 halo: tile starts at (block_i - 1, block_j - 1)
        __shared__ float s_Hx[(BX+1) * (BY+1)];
        __shared__ float s_Hy[(BX+1) * (BY+1)];

        int tx = threadIdx.x;
        int ty = threadIdx.y;
        int i = blockIdx.x * BX + tx;
        int j = blockIdx.y * BY + ty;

        int sp = BY + 1;  // shared memory pitch

        // Load Hx[i, j] into s_Hx[tx+1, ty+1] (offset by 1 for the -1 halo)
        // Main tile
        if (i < Nx && j < Ny) {
            int idx = i * Ny + j;
            s_Hx[(tx+1) * sp + (ty+1)] = Hx[idx];
            s_Hy[(tx+1) * sp + (ty+1)] = Hy[idx];
        } else {
            s_Hx[(tx+1) * sp + (ty+1)] = 0.0f;
            s_Hy[(tx+1) * sp + (ty+1)] = 0.0f;
        }

        // Load -1 halo in j (left column: ty==0 loads j-1)
        if (ty == 0) {
            int jj = j - 1;
            if (i < Nx && jj >= 0) {
                s_Hx[(tx+1) * sp + 0] = Hx[i * Ny + jj];
                s_Hy[(tx+1) * sp + 0] = Hy[i * Ny + jj];
            } else {
                s_Hx[(tx+1) * sp + 0] = 0.0f;
                s_Hy[(tx+1) * sp + 0] = 0.0f;
            }
        }

        // Load -1 halo in i (top row: tx==0 loads i-1)
        if (tx == 0) {
            int ii = i - 1;
            if (ii >= 0 && j < Ny) {
                s_Hx[0 * sp + (ty+1)] = Hx[ii * Ny + j];
                s_Hy[0 * sp + (ty+1)] = Hy[ii * Ny + j];
            } else {
                s_Hx[0 * sp + (ty+1)] = 0.0f;
                s_Hy[0 * sp + (ty+1)] = 0.0f;
            }
        }

        __syncthreads();

        if (i >= Nx || j >= Ny) return;
        int idx = i * Ny + j;

        // Ez update from shared memory
        if (i >= 1 && j >= 1) {
            float dHy = (s_Hy[(tx+1)*sp + (ty+1)] - s_Hy[tx*sp + (ty+1)]) / dx;
            float dHx = (s_Hx[(tx+1)*sp + (ty+1)] - s_Hx[(tx+1)*sp + ty])   / dy;
            Ez[idx] += coeff_ez * (dHy - dHx);
        }

        // Source + PEC (same as global version)
        if (i == src_x && j == src_y)
            Ez[idx] = pulse_arr[step];
        if (i == 0 || i == Nx-1 || j == 0 || j == Ny-1)
            Ez[idx] = 0.0f;
    }
    ''', 'update_Ez_shared')

    print('Shared memory kernels compiled.')
    print(f'  H kernel shared mem : {(8+1)*(32+1)*4} bytes = {(8+1)*(32+1)*4/1024:.1f} KB (1 tile)')
    print(f'  Ez kernel shared mem: {2*(8+1)*(32+1)*4} bytes = {2*(8+1)*(32+1)*4/1024:.1f} KB (2 tiles)')

## Benchmark: Global vs Shared Memory (CuPy)

In [ ]:
if not HAS_CUPY:
    print('Skipping CuPy benchmark (no CuPy)')
    t_global = t_shared = None
else:
    # ═══════════════════════════════════════════════════════════════
    #  Benchmark helper
    # ═══════════════════════════════════════════════════════════════

    _dx = np.float32(dx)
    _dy = np.float32(dy)
    _Nx = np.int32(Nx)
    _Ny = np.int32(Ny)
    _sx = np.int32(src_x)
    _sy = np.int32(src_y)

    def run_cupy_benchmark(H_kernel, Ez_kernel, label, n_warmup=10):
        """Run FDTD with given kernels, return elapsed time."""
        Ez = cp.zeros((Nx, Ny), dtype=cp.float32)
        Hx = cp.zeros((Nx, Ny), dtype=cp.float32)
        Hy = cp.zeros((Nx, Ny), dtype=cp.float32)

        # Warmup
        for _ in range(n_warmup):
            H_kernel(GRID, BLOCK, (Ez, Hx, Hy, coeff_hx, coeff_hy, _Nx, _Ny))
            Ez_kernel(GRID, BLOCK, (Ez, Hx, Hy, coeff_ez, _dx, _dy,
                                    _Nx, _Ny, _sx, _sy, pulse_gpu, np.int32(0)))
        cp.cuda.Device().synchronize()

        # Reset
        Ez[:] = 0; Hx[:] = 0; Hy[:] = 0

        # Timed run
        cp.cuda.Device().synchronize()
        t0 = time.time()
        for n in range(n_steps):
            H_kernel(GRID, BLOCK, (Ez, Hx, Hy, coeff_hx, coeff_hy, _Nx, _Ny))
            Ez_kernel(GRID, BLOCK, (Ez, Hx, Hy, coeff_ez, _dx, _dy,
                                    _Nx, _Ny, _sx, _sy, pulse_gpu, np.int32(n)))
        cp.cuda.Device().synchronize()
        elapsed = time.time() - t0

        gflops = flops_total / elapsed / 1e9
        print(f'  {label:<24} {elapsed*1000:>8.2f} ms  '
              f'({elapsed/n_steps*1000:.4f} ms/step)  '
              f'{gflops:.2f} GFLOP/s')
        return elapsed, Ez


    print(f'CuPy Benchmark ({Nx}x{Ny}, {n_steps} steps)')
    print(f'{"-"*72}')

    t_global, Ez_global = run_cupy_benchmark(
        cupy_H_global, cupy_Ez_global, 'Global Memory')

    t_shared, Ez_shared = run_cupy_benchmark(
        cupy_H_shared, cupy_Ez_shared, 'Shared Memory')

    print(f'{"-"*72}')
    speedup = t_global / t_shared
    print(f'  Speedup: {speedup:.2f}x {"faster" if speedup > 1 else "slower"} with shared memory')

    # Verify results match
    diff = float(cp.max(cp.abs(Ez_global - Ez_shared)))
    rel  = diff / max(float(cp.max(cp.abs(Ez_global))), 1e-30)
    print(f'  Max |diff|: {diff:.2e}, relative: {rel:.2e}',
          ' (MATCH)' if rel < 1e-5 else ' (MISMATCH!)')

## NumPy Reference (CPU baseline for TFLOP/s context)

In [ ]:
print(f'NumPy Benchmark ({Nx}x{Ny}, {n_steps} steps)')

Ez_np = np.zeros((Nx, Ny), dtype=np.float64)
Hx_np = np.zeros((Nx, Ny), dtype=np.float64)
Hy_np = np.zeros((Nx, Ny), dtype=np.float64)

t_np_start = time.time()
for n in range(n_steps):
    t = n * dt
    Hx_np[:, :-1] -= float(coeff_hx) * (Ez_np[:, 1:] - Ez_np[:, :-1])
    Hy_np[:-1, :] += float(coeff_hy) * (Ez_np[1:, :] - Ez_np[:-1, :])
    Ez_np[1:, 1:] += float(coeff_ez) * (
        (Hy_np[1:, 1:] - Hy_np[:-1, 1:]) / dx
      - (Hx_np[1:, 1:] - Hx_np[1:, :-1]) / dy
    )
    Ez_np[src_x, src_y] = np.exp(-((t - t0) / spread) ** 2)
    Ez_np[0, :] = 0; Ez_np[-1, :] = 0
    Ez_np[:, 0] = 0; Ez_np[:, -1] = 0
t_np = time.time() - t_np_start

gflops_np = flops_total / t_np / 1e9
print(f'  NumPy:  {t_np:.4f} s  ({t_np/n_steps*1000:.3f} ms/step)  {gflops_np:.2f} GFLOP/s')

## CUDA C Versions (%%writefile + nvcc)

Both global and shared memory kernels in a single `.cu` file, benchmarked back-to-back with `cudaEvent` timing.

In [ ]:
%%writefile fdtd_shared_bench.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e = call; if(e!=cudaSuccess) { fprintf(stderr,"CUDA %s:%d: %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); exit(1); } } while(0)

#define NX 1000
#define NY 1000
#define N_STEPS 100
#define DX 1.0e-3f
#define DY 1.0e-3f
#define C0 3.0e8f
#define MU0 1.2566370614359173e-6f
#define EPS0 8.854187817620389e-12f
#define BX 8
#define BY 32

/* ═══ Global Memory Kernels ═══ */

__global__ void H_global(float* __restrict__ Ez, float* __restrict__ Hx,
                         float* __restrict__ Hy, float chx, float chy, int Nx, int Ny) {
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int j = blockIdx.y*blockDim.y + threadIdx.y;
    if (i < Nx && j < Ny-1) { int idx=i*Ny+j; Hx[idx] -= chx*(Ez[idx+1]-Ez[idx]); }
    if (i < Nx-1 && j < Ny) { int idx=i*Ny+j; Hy[idx] += chy*(Ez[idx+Ny]-Ez[idx]); }
}

__global__ void Ez_global(float* __restrict__ Ez, const float* __restrict__ Hx,
                          const float* __restrict__ Hy, float cez, float dx, float dy,
                          int Nx, int Ny, int sx, int sy, float pulse) {
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int j = blockIdx.y*blockDim.y + threadIdx.y;
    if (i>=Nx||j>=Ny) return;
    int idx = i*Ny+j;
    if (i>=1&&j>=1) Ez[idx] += cez*((Hy[idx]-Hy[idx-Ny])/dx - (Hx[idx]-Hx[idx-1])/dy);
    if (i==sx&&j==sy) Ez[idx]=pulse;
    if (i==0||i==Nx-1||j==0||j==Ny-1) Ez[idx]=0.0f;
}

/* ═══ Shared Memory Kernels ═══ */

__global__ void H_shared(float* Ez, float* Hx, float* Hy,
                         float chx, float chy, int Nx, int Ny) {
    __shared__ float s_Ez[(BX+1)*(BY+1)];
    int tx=threadIdx.x, ty=threadIdx.y;
    int i=blockIdx.x*BX+tx, j=blockIdx.y*BY+ty;
    int sp=BY+1;

    s_Ez[tx*sp+ty] = (i<Nx&&j<Ny) ? Ez[i*Ny+j] : 0.0f;
    if (ty==BY-1||j==Ny-1) {
        int jj=j+1;
        s_Ez[tx*sp+ty+1] = (i<Nx&&jj<Ny) ? Ez[i*Ny+jj] : 0.0f;
    }
    if (tx==BX-1||i==Nx-1) {
        int ii=i+1;
        s_Ez[(tx+1)*sp+ty] = (ii<Nx&&j<Ny) ? Ez[ii*Ny+j] : 0.0f;
    }
    __syncthreads();

    if (i<Nx&&j<Ny-1) { int idx=i*Ny+j; Hx[idx]-=chx*(s_Ez[tx*sp+ty+1]-s_Ez[tx*sp+ty]); }
    if (i<Nx-1&&j<Ny) { int idx=i*Ny+j; Hy[idx]+=chy*(s_Ez[(tx+1)*sp+ty]-s_Ez[tx*sp+ty]); }
}

__global__ void Ez_shared(float* Ez, const float* Hx, const float* Hy,
                          float cez, float dx, float dy,
                          int Nx, int Ny, int sx, int sy, float pulse) {
    __shared__ float s_Hx[(BX+1)*(BY+1)];
    __shared__ float s_Hy[(BX+1)*(BY+1)];
    int tx=threadIdx.x, ty=threadIdx.y;
    int i=blockIdx.x*BX+tx, j=blockIdx.y*BY+ty;
    int sp=BY+1;

    if (i<Nx&&j<Ny) {
        int idx=i*Ny+j;
        s_Hx[(tx+1)*sp+(ty+1)]=Hx[idx];
        s_Hy[(tx+1)*sp+(ty+1)]=Hy[idx];
    } else {
        s_Hx[(tx+1)*sp+(ty+1)]=0; s_Hy[(tx+1)*sp+(ty+1)]=0;
    }
    if (ty==0) {
        int jj=j-1;
        if (i<Nx&&jj>=0) { s_Hx[(tx+1)*sp]=Hx[i*Ny+jj]; s_Hy[(tx+1)*sp]=Hy[i*Ny+jj]; }
        else { s_Hx[(tx+1)*sp]=0; s_Hy[(tx+1)*sp]=0; }
    }
    if (tx==0) {
        int ii=i-1;
        if (ii>=0&&j<Ny) { s_Hx[0*sp+(ty+1)]=Hx[ii*Ny+j]; s_Hy[0*sp+(ty+1)]=Hy[ii*Ny+j]; }
        else { s_Hx[0*sp+(ty+1)]=0; s_Hy[0*sp+(ty+1)]=0; }
    }
    __syncthreads();

    if (i>=Nx||j>=Ny) return;
    int idx=i*Ny+j;
    if (i>=1&&j>=1) {
        float dHy=(s_Hy[(tx+1)*sp+(ty+1)]-s_Hy[tx*sp+(ty+1)])/dx;
        float dHx=(s_Hx[(tx+1)*sp+(ty+1)]-s_Hx[(tx+1)*sp+ty])/dy;
        Ez[idx]+=cez*(dHy-dHx);
    }
    if (i==sx&&j==sy) Ez[idx]=pulse;
    if (i==0||i==Nx-1||j==0||j==Ny-1) Ez[idx]=0.0f;
}

/* ═══ Main ═══ */

int main() {
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s  (%d SMs, %zu MB VRAM)\n", prop.name, prop.multiProcessorCount, prop.totalGlobalMem/(1024*1024));

    int Nx=NX, Ny=NY;
    float dt_ = 0.99f/(C0*sqrtf(1.0f/(DX*DX)+1.0f/(DY*DY)));
    float spread=12.0f*dt_, t0_=3.0f*spread;
    float chx=dt_/(MU0*DY), chy=dt_/(MU0*DX), cez=dt_/EPS0;
    int sx=Nx/2, sy=Ny/2;
    size_t bytes=(size_t)Nx*Ny*sizeof(float);

    float *d_Ez, *d_Hx, *d_Hy;
    CUDA_CHECK(cudaMalloc(&d_Ez,bytes));
    CUDA_CHECK(cudaMalloc(&d_Hx,bytes));
    CUDA_CHECK(cudaMalloc(&d_Hy,bytes));

    dim3 block(BX,BY);
    dim3 grid((Nx+BX-1)/BX,(Ny+BY-1)/BY);

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    printf("\nGrid: %dx%d, %d steps, block(%d,%d)\n\n", Nx, Ny, N_STEPS, BX, BY);

    // ── Global Memory ──
    CUDA_CHECK(cudaMemset(d_Ez,0,bytes)); CUDA_CHECK(cudaMemset(d_Hx,0,bytes)); CUDA_CHECK(cudaMemset(d_Hy,0,bytes));
    // warmup
    for(int n=0;n<10;n++) { H_global<<<grid,block>>>(d_Ez,d_Hx,d_Hy,chx,chy,Nx,Ny); Ez_global<<<grid,block>>>(d_Ez,d_Hx,d_Hy,cez,DX,DY,Nx,Ny,sx,sy,0.0f); }
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemset(d_Ez,0,bytes)); CUDA_CHECK(cudaMemset(d_Hx,0,bytes)); CUDA_CHECK(cudaMemset(d_Hy,0,bytes));

    CUDA_CHECK(cudaEventRecord(start));
    for(int n=0;n<N_STEPS;n++) {
        float t=n*dt_;
        float p=expf(-((t-t0_)/spread)*((t-t0_)/spread));
        H_global<<<grid,block>>>(d_Ez,d_Hx,d_Hy,chx,chy,Nx,Ny);
        Ez_global<<<grid,block>>>(d_Ez,d_Hx,d_Hy,cez,DX,DY,Nx,Ny,sx,sy,p);
    }
    CUDA_CHECK(cudaEventRecord(stop)); CUDA_CHECK(cudaEventSynchronize(stop));
    float ms_global=0; CUDA_CHECK(cudaEventElapsedTime(&ms_global,start,stop));
    printf("  Global Memory : %8.3f ms  (%.4f ms/step)\n", ms_global, ms_global/N_STEPS);

    // Save global result for verification
    float *h_Ez_global = (float*)malloc(bytes);
    CUDA_CHECK(cudaMemcpy(h_Ez_global,d_Ez,bytes,cudaMemcpyDeviceToHost));

    // ── Shared Memory ──
    CUDA_CHECK(cudaMemset(d_Ez,0,bytes)); CUDA_CHECK(cudaMemset(d_Hx,0,bytes)); CUDA_CHECK(cudaMemset(d_Hy,0,bytes));
    // warmup
    for(int n=0;n<10;n++) { H_shared<<<grid,block>>>(d_Ez,d_Hx,d_Hy,chx,chy,Nx,Ny); Ez_shared<<<grid,block>>>(d_Ez,d_Hx,d_Hy,cez,DX,DY,Nx,Ny,sx,sy,0.0f); }
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemset(d_Ez,0,bytes)); CUDA_CHECK(cudaMemset(d_Hx,0,bytes)); CUDA_CHECK(cudaMemset(d_Hy,0,bytes));

    CUDA_CHECK(cudaEventRecord(start));
    for(int n=0;n<N_STEPS;n++) {
        float t=n*dt_;
        float p=expf(-((t-t0_)/spread)*((t-t0_)/spread));
        H_shared<<<grid,block>>>(d_Ez,d_Hx,d_Hy,chx,chy,Nx,Ny);
        Ez_shared<<<grid,block>>>(d_Ez,d_Hx,d_Hy,cez,DX,DY,Nx,Ny,sx,sy,p);
    }
    CUDA_CHECK(cudaEventRecord(stop)); CUDA_CHECK(cudaEventSynchronize(stop));
    float ms_shared=0; CUDA_CHECK(cudaEventElapsedTime(&ms_shared,start,stop));
    printf("  Shared Memory : %8.3f ms  (%.4f ms/step)\n", ms_shared, ms_shared/N_STEPS);

    // Save shared result
    float *h_Ez_shared = (float*)malloc(bytes);
    CUDA_CHECK(cudaMemcpy(h_Ez_shared,d_Ez,bytes,cudaMemcpyDeviceToHost));

    // ── Results ──
    float speedup = ms_global / ms_shared;
    printf("\n  Speedup: %.2fx %s with shared memory\n", speedup, speedup>1?"faster":"slower");

    // GFLOP/s
    long long flops = (long long)Nx*(Ny-1)*3 + (long long)(Nx-1)*Ny*3 + (long long)(Nx-1)*(Ny-1)*7;
    flops *= N_STEPS;
    printf("  Global GFLOP/s: %.2f\n", (float)flops/ms_global/1e6);
    printf("  Shared GFLOP/s: %.2f\n", (float)flops/ms_shared/1e6);

    // Verification
    float maxdiff = 0;
    for(int k=0;k<Nx*Ny;k++) { float d=fabsf(h_Ez_global[k]-h_Ez_shared[k]); if(d>maxdiff) maxdiff=d; }
    printf("  Max |diff|: %.2e %s\n", maxdiff, maxdiff<1e-5?"(MATCH)":"(MISMATCH!)");

    cudaFree(d_Ez); cudaFree(d_Hx); cudaFree(d_Hy);
    free(h_Ez_global); free(h_Ez_shared);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

In [ ]:
!nvcc -O3 -o fdtd_shared_bench fdtd_shared_bench.cu -lm && ./fdtd_shared_bench

## Summary

In [ ]:
print(f"{'='*72}")
print(f"  SHARED MEMORY EXPERIMENT — {Nx}x{Ny} grid, {n_steps} steps")
print(f"  Working set: {working_set/1024/1024:.1f} MB (exceeds 4 MB L2 cache)")
print(f"{'='*72}")
print(f"")
print(f"  {'Version':<28} {'Time':>10} {'ms/step':>10} {'GFLOP/s':>10}")
print(f"  {'-'*28} {'-'*10} {'-'*10} {'-'*10}")

results = [('NumPy (CPU)', t_np)]
if t_global is not None:
    results.append(('CuPy Global Memory', t_global))
if t_shared is not None:
    results.append(('CuPy Shared Memory', t_shared))

for name, t in results:
    ms = t * 1000
    mss = t / n_steps * 1000
    gf = flops_total / t / 1e9
    print(f"  {name:<28} {ms:>8.2f} ms {mss:>9.4f}  {gf:>9.2f}")

print(f"")
if t_global and t_shared:
    print(f"  CuPy: Shared vs Global = {t_global/t_shared:.2f}x")
    print(f"  CuPy Global vs NumPy   = {t_np/t_global:.1f}x")
    print(f"  CuPy Shared vs NumPy   = {t_np/t_shared:.1f}x")
else:
    print(f"  (CuPy unavailable — see CUDA C results above)")
print(f"")
print(f"  (CUDA C results above from nvcc binary with cudaEvent timing)")
print(f"{'='*72}")

## Analysis

### Why shared memory helps at 1000x1000
- Working set = **11.7 MB** >> L2 cache (4 MB)
- Global memory reads now hit **DRAM** (200-900 GB/s), not L2 cache (2-4 TB/s)
- Shared memory (~100 TB/s effective) eliminates redundant DRAM fetches
- Each Ez value is read by 2 H-threads and each H value by 2 Ez-threads — shared memory avoids the duplicate DRAM load

### Why shared memory didn't help at 200x200
- Working set = **469 KB** << L2 cache (4 MB)
- Global reads were already served from L2 at near-shared-memory speed
- `__syncthreads()` overhead was larger than the memory savings

### The stencil reuse is still limited
- FDTD nearest-neighbor stencil: each value reused by only 2 threads
- For higher-order stencils (wider neighborhood), shared memory gains would be larger
- The main benefit here is avoiding duplicate DRAM fetches, not massive reuse